# 04 — Pretrained CNNs in Rust: ResNet18, VGG19, and MobileNetV2

**Goal:** download **real** checkpoints compatible with [tch-rs](https://github.com/LaurentMazare/tch-rs), verify the files, and explain *transfer learning*, *fine-tuning*, and *deep fine-tuning*. **We do not train** any of these models.

This notebook uses the **Rust / Evcxr** kernel. The executed cells use Rust's standard library and `curl` so the *easy setup* does not require libtorch. For an optional Rust example that **loads model tensors and actually changes `requires_grad`** using tch/libtorch, see [`examples/cnn_finetuning/src/main.rs`](../examples/cnn_finetuning/src/main.rs). **Downloading and checking checkpoint files is not the same as loading them into tch or running inference.**

⚠️ **Large downloads:** ResNet18 ≈47 MB, MobileNetV2 ≈14 MB, and VGG19 ≈575 MB. Files are stored in a temporary directory outside Git. You need an Internet connection, `curl`, enough disk space, and must comply with applicable checkpoint licenses and terms. URLs refer to the assets from the [tch-rs `mw` release](https://github.com/LaurentMazare/tch-rs/releases/tag/mw).

## 1. Catalog of Rust-compatible checkpoints

Do not confuse model architectures with weights: **`.ot`** files contain weights exported to a format that `tch::nn::VarStore::load` can read. Torchvision `.pth` files cannot simply be loaded as `.ot` files.

In [1]:
let checkpoints: [(&str, &str, u64); 3] = [
    ("ResNet18", "resnet18.ot", 46_831_783),
    ("MobileNetV2", "mobilenet-v2.ot", 14_249_059),
    ("VGG19", "vgg19.ot", 574_678_008),
];
for (name, file, bytes) in checkpoints {
    println!("{name:<12} {file:<18} {bytes:>10} bytes ({:.1} MiB)", bytes as f64 / 1_048_576.0);
}
println!("Source: https://github.com/LaurentMazare/tch-rs/releases/tag/mw");

ResNet18     resnet18.ot          46831783 bytes (44.7 MiB)


MobileNetV2  mobilenet-v2.ot      14249059 bytes (13.6 MiB)


VGG19        vgg19.ot            574678008 bytes (548.1 MiB)


Source: https://github.com/LaurentMazare/tch-rs/releases/tag/mw


## 2. Download all three checkpoints from Rust

Use `std::process::Command` to invoke `curl --fail --location`, then verify file sizes to detect incomplete downloads. **Weights are never committed to the repository.** Later runs reuse previously downloaded files if the sizes match. In GitHub Actions, the weights live only on the temporary runner; the notebook commits **text outputs**, not model files.

In [2]:
let cache_dir = std::env::temp_dir().join("rust-for-ai-cnn-weights");
std::fs::create_dir_all(&cache_dir).expect("Cannot create model cache directory");
let checkpoints: [(&str, &str, u64); 3] = [
    ("ResNet18", "resnet18.ot", 46_831_783),
    ("MobileNetV2", "mobilenet-v2.ot", 14_249_059),
    ("VGG19", "vgg19.ot", 574_678_008),
];
if std::env::var("RUST_FOR_AI_CNN_OFFLINE").as_deref() == Ok("1") {
    println!("Offline Rust syntax check: downloads skipped (CI compile/run smoke test).");
} else {
for (name, file, expected) in checkpoints {
    let destination = cache_dir.join(file);
    let found = std::fs::metadata(&destination).map(|x| x.len()).unwrap_or(0);
    if found != expected {
        let temp = destination.with_extension("ot.part");
        let url = format!("https://github.com/LaurentMazare/tch-rs/releases/download/mw/{file}");
        let status = std::process::Command::new("curl")
            .args(["--fail", "--location", "--silent", "--show-error", "--retry", "2", "--connect-timeout", "30", "--max-time", "600", "--output"])
            .arg(&temp).arg(&url).status().expect("curl must be installed");
        assert!(status.success(), "Download failed for {name}: {url}");
        let actual = std::fs::metadata(&temp).expect("Missing download").len();
        assert_eq!(actual, expected, "Truncated checkpoint: {name}");
        std::fs::rename(&temp, &destination).expect("Cannot move downloaded model");
    }
    let actual = std::fs::metadata(&destination).unwrap().len();
    assert_eq!(actual, expected);
    println!("OK: {name} checkpoint downloaded/verified ({actual} bytes)");
}
println!("Downloaded checkpoints: 3/3; weights have NOT been loaded as tensors.");
}

OK: ResNet18 checkpoint downloaded/verified (46831783 bytes)


OK: MobileNetV2 checkpoint downloaded/verified (14249059 bytes)


OK: VGG19 checkpoint downloaded/verified (574678008 bytes)


Downloaded checkpoints: 3/3; weights have NOT been loaded as tensors.


()

## 3. Three fine-tuning strategies in Rust

**Linear probing / head-only:** freeze the entire pretrained backbone and train **only the new classification head**. **Deep fine-tuning:** unfreeze the final backbone blocks as well as the head. **Full fine-tuning:** unfreeze all parameters, usually with a smaller learning rate.

The example in [`examples/cnn_finetuning/src/main.rs`](../examples/cnn_finetuning/src/main.rs) demonstrates `VarStore::load`, `VarStore::freeze`, `Tensor::set_requires_grad`, `VarStore::unfreeze`, and an `nn::linear` output layer. **Important:** for a common example across all three architectures, this adds a trainable adapter on top of the pretrained model's **1,000 ImageNet logits**. This is **not** the same as replacing the original classifier with a head fed by penultimate features. For stronger transfer learning, using backbone features and replacing the original classifier is often preferable.

This optional variant requires **tch and a compatible libtorch/PyTorch installation**. They are not included in the basic setup, and the heavier variant is not executed by the automatic notebook workflow.

In [3]:
let modes = [
    ("head", "frozen backbone + new classification head"),
    ("deep", "trainable head and final backbone block"),
    ("full", "all parameters trainable"),
];
for (mode, detail) in modes {
    println!("{mode:<5}: {detail}");
}
println!("See the runnable tch example: examples/cnn_finetuning/src/main.rs");
println!("Training performed in this notebook: NONE.");

head : frozen backbone + new classification head


deep : trainable head and final backbone block


full : all parameters trainable


See the runnable tch example: examples/cnn_finetuning/src/main.rs


Training performed in this notebook: NONE.


## 4. Try real parameter manipulation with `tch` (optional)

After running this notebook, the weights will be available in your machine's temporary cache. Install **PyTorch/libtorch compatible with the pinned `tch` version**, then run these commands from the repository root:

```bash
# Follow https://github.com/LaurentMazare/tch-rs for compatible libtorch/PyTorch setup.
# For example, if a compatible PyTorch is installed in your environment:
export LIBTORCH_USE_PYTORCH=1
cargo run --manifest-path examples/cnn_finetuning/Cargo.toml -- resnet18 head 3
cargo run --manifest-path examples/cnn_finetuning/Cargo.toml -- mobilenet-v2 deep 3
cargo run --manifest-path examples/cnn_finetuning/Cargo.toml -- vgg19 full 3
```

These commands construct each architecture, load its `.ot` checkpoint, add a **three-class adapter head**, and change the `requires_grad` flags. **They do not run a training epoch or update model weights.** The executable reads from the same local temporary cache; on another machine, run this notebook's download step again.

To train later, you will need labeled images, ImageNet-style preprocessing (RGB conversion, resizing, and normalization), a loss function, and an optimizer. Do not report accuracy for a new task without training and evaluating against actual data.